### Importy

In [1]:
import os
import sys
import subprocess
import random
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from tqdm.auto import tqdm
from transformers import AutoImageProcessor, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans, AgglomerativeClustering

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import math

from pathlib import Path

In [2]:
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark     = True

### Konfiguracja notebooka

In [ ]:
# --- SETUP ---
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
warnings.filterwarnings("ignore", message=".*Torch was not compiled with flash attention.*")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 45390
PROJECT_PATH = os.path.abspath(".")
print(f"Device: {device}")
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

# --- KONFIGURACJA DATASETU ---
DATASET = "alpha_10_seed_129"  # zmień na właściwy

DATASET_CONFIGS = {
    "alpha_10_seed_129": {
        "dir_name":       r"places365_10_classes\129\alpha_1",
        "train_dir":      "train",
        "val_dir":        "valid",
        "test_dir":       "test",
        "split_val_test": False,
        "num_classes":    10,
        "alpha":          1,
    },
    "skewed_places365_alpha10_2": {
        "dir_name":       "skewed_places365_alpha1_2",
        "train_dir":      "train",
        "val_dir":        "valid",
        "test_dir":       "test",
        "split_val_test": False,
        "num_classes":    10,
        "alpha":          1,
    },
    "skewed_imagenet1k_alpha05": {
        "dir_name":       "skewed_imagenet1k_alpha05",
        "train_dir":      "train",
        "val_dir":        "val",
        "test_dir":       "test",
        "split_val_test": False,
        "num_classes":    10,
        "alpha":          0.5,
    },
    "skewed_places365_alpha10": {
        "dir_name":       "skewed_places365_alpha1",
        "train_dir":      "train",
        "val_dir":        "val",
        "test_dir":       None,
        "split_val_test": False,
        "num_classes":    10,
        "alpha":          1.0,
    },
    "skewed_places365_alpha05": {
        "dir_name":       "skewed_places365_alpha05",
        "train_dir":      "train",
        "val_dir":        "val",
        "test_dir":       None,
        "split_val_test": True,
        "num_classes":    10,
        "alpha":          0.5,
    },
    "skewed_places365_alpha01": {
        "dir_name":       "skewed_places365_alpha01",
        "train_dir":      "train",
        "val_dir":        "val",
        "test_dir":       None,
        "split_val_test": True,
        "num_classes":    10,
        "alpha":          0.1,
    },
}

DATA_ROOT = os.path.join(PROJECT_PATH, "data")
_cfg        = DATASET_CONFIGS[DATASET]
NUM_CLASSES = _cfg["num_classes"]
DATA_PATH   = os.path.join(DATA_ROOT, _cfg["dir_name"])

print(f"Dataset : {DATASET}  (α={_cfg['alpha']})")
print(f"Klasy   : {NUM_CLASSES}")
print(f"Ścieżka : {DATA_PATH}")

# --- REPO Z PAPIERU ---
REPO_NAME = "ssl-data-curation"
REPO_URL  = "https://github.com/facebookresearch/ssl-data-curation.git"
REPO_PATH = os.path.join(PROJECT_PATH, f"{REPO_NAME}/")

# --- HIPERPARAMETRY TRENINGU OD ZERA ---
MAX_EPOCHS              = 150
EARLY_STOPPING_PATIENCE = 20   
BATCH_SIZE              = 64
LR                      = 0.1
MOMENTUM                = 0.9
WEIGHT_DECAY            = 1e-4

# --- METODY SAMPLOWANIA DO PORÓWNANIA ---
SAMPLING_METHODS = ["hierarchical", "random", "agglomerative", "kmeans"]
SEEDS = [45390, 12345, 98765,1081]
percentages   = [0.3, 0.5, 0.7, 0.9]
# --- PARAMETRY HIERARCHICZNEGO K-MEANS (DINO) ---
N_CLUSTERS_DINO   = [40, 10]
N_LEVELS_DINO     = 2
SAMPLE_SIZES_DINO = [400, 1600]
# cofig kacpera z optuny

# N_CLUSTERS_DINO = [183, 116, 14]
# N_LEVELS_DINO = 3
# SAMPLE_SIZES_DINO =  [100, 245, 969]

# --- EMBEDDINGI DINO: ładowanie z pliku lub ekstrakcja na żywo ---
# True - wczytaj gotowe embeddingi z pliku .pkl (szybko, bez GPU dla ekstrakcji)
# False - oblicz embeddingi na żywo z modelu DINOv2-large
LOAD_DINO_EMBEDDINGS_FROM_PKL = True
DINO_EMBEDDINGS_PKL_PATH = os.path.join(PROJECT_PATH, "dino_embeddings1.0_places365.pt")


print(f"Ładowanie embeddingów z pkl: {LOAD_DINO_EMBEDDINGS_FROM_PKL}")
if LOAD_DINO_EMBEDDINGS_FROM_PKL:
    print(f"Ścieżka pkl: {DINO_EMBEDDINGS_PKL_PATH}")


Device: cuda
Dataset : alpha_10_seed_129  (α=1)
Klasy   : 10
Ścieżka : c:\Users\Kacper\Documents\iad 6 sem\wb2\WB2_Project\data\places365_10_classes\129\alpha_1
Ładowanie embeddingów z pkl: True
Ścieżka pkl: c:\Users\Kacper\Documents\iad 6 sem\wb2\WB2_Project\dino_embeddings1.0_places365.pt


In [4]:
def setup_repo():
    if not os.path.exists(REPO_PATH):
        print(f"--- Klonowanie repozytorium {REPO_NAME}... ---")
        subprocess.run(["git", "clone", REPO_URL], check=True)
    src_path = os.path.join(REPO_PATH, "src")
    if src_path not in sys.path:
        sys.path.insert(0, src_path)
    if REPO_PATH not in sys.path:
        sys.path.insert(0, REPO_PATH)
    print("--- Repository paths configured ---")

setup_repo()
from src.clusters import HierarchicalCluster
from src import hierarchical_kmeans_gpu as hkmg
from src import hierarchical_sampling as hs
from other_clustering_methods import kmeans_sampling, dbscan_sampling
from other_clustering_methods import _sample_from_labels


--- Repository paths configured ---


### Wczytanie datasetu

In [5]:
class ResnetTransform:
    """Standardowy transform do ekstrakcji cech i ewaluacji."""
    def __init__(self):
        self.transform = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
    def __call__(self, img):
        return self.transform(img)


# Augmentacja treningowa — kluczowa przy treningu od zera.
# Używana TYLKO podczas treningu modelu, nie przy ekstrakcji embeddingów.
scratch_train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.2, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


class DinoTransform:
    def __init__(self, processor=AutoImageProcessor.from_pretrained('facebook/dinov2-large')):
        self.processor = processor
    def __call__(self, img):
        return self.processor(images=img, return_tensors="pt")["pixel_values"].squeeze(0)


def _split_val_test(base_dataset, seed: int):
    indices = np.arange(len(base_dataset))
    v_idx, t_idx = train_test_split(
        indices, test_size=0.5, stratify=base_dataset.targets, random_state=seed
    )
    return Subset(base_dataset, v_idx), Subset(base_dataset, t_idx)


def load_datasets(cfg: dict, data_path: str, seed: int, batch_size: int = 64):
    train_dir = os.path.join(data_path, cfg["train_dir"])

    # Dwa widoki train setu:
    # - ResnetTransform: do ekstrakcji cech i samplowania
    # - DinoTransform:   do ekstrakcji embeddingów DINO
    # - scratch_train_transform: używany PODCZAS treningu modelu od zera
    train_dataset_resnet = datasets.ImageFolder(train_dir, transform=ResnetTransform())
    train_dataset_dino   = datasets.ImageFolder(train_dir, transform=DinoTransform())
    # Osobna instancja datasetu z augmentacją treningową — używana przy DataLoaderze do treningu
    train_dataset_scratch = datasets.ImageFolder(train_dir, transform=scratch_train_transform)

    if cfg["split_val_test"]:
        if cfg["val_dir"] is None and cfg["test_dir"] is None:
            # Brak val i test - dzielimy train na 70/15/15
            all_indices = np.arange(len(train_dataset_resnet))
            all_targets = train_dataset_resnet.targets

            train_idx, valtest_idx = train_test_split(
                all_indices, test_size=0.05, stratify=all_targets, random_state=seed
            )
            valtest_targets = [all_targets[i] for i in valtest_idx]
            val_idx, test_idx = train_test_split(
                valtest_idx, test_size=0.5, stratify=valtest_targets, random_state=seed
            )

            train_dataset_resnet  = Subset(train_dataset_resnet, train_idx)
            train_dataset_dino    = Subset(train_dataset_dino, train_idx)
            train_dataset_scratch = Subset(train_dataset_scratch, train_idx)

            val_dataset = Subset(
                datasets.ImageFolder(train_dir, transform=ResnetTransform()), val_idx
            )
            test_dataset = Subset(
                datasets.ImageFolder(train_dir, transform=ResnetTransform()), test_idx
            )

        elif cfg["test_dir"] is None:
            # Jest val, brak test - dzielimy val 80/20
            val_base = datasets.ImageFolder(
                os.path.join(data_path, cfg["val_dir"]), transform=ResnetTransform()
            )
            indices = np.arange(len(val_base))
            v_idx, t_idx = train_test_split(
                indices, test_size=0.05, stratify=val_base.targets, random_state=seed
            )
            val_dataset  = Subset(val_base, v_idx)
            test_dataset = Subset(val_base, t_idx)

        elif cfg["val_dir"] is None:
            # Jest test, brak val - dzielimy test 80/20
            test_base = datasets.ImageFolder(
                os.path.join(data_path, cfg["test_dir"]), transform=ResnetTransform()
            )
            indices = np.arange(len(test_base))
            t_idx, v_idx = train_test_split(
                indices, test_size=0.05, stratify=test_base.targets, random_state=seed
            )
            test_dataset = Subset(test_base, t_idx)
            val_dataset  = Subset(test_base, v_idx)

    else:
        if cfg["val_dir"] is not None:
            val_dataset = datasets.ImageFolder(
                os.path.join(data_path, cfg["val_dir"]), transform=ResnetTransform()
            )
        if cfg["test_dir"] is not None:
            test_dataset = datasets.ImageFolder(
                os.path.join(data_path, cfg["test_dir"]), transform=ResnetTransform()
            )

   
    print(f"Train : {len(train_dataset_resnet):>6} próbek")
    if cfg["val_dir"] is not None or cfg["split_val_test"]:
        print(f"Val   : {len(val_dataset):>6} próbek")
        val_loader  = DataLoader(val_dataset,  batch_size=batch_size, shuffle=False, num_workers=0)
    else:
        val_loader = None

    if cfg["test_dir"] is not None or cfg["split_val_test"]:
        print(f"Test  : {len(test_dataset):>6} próbek")
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    else:
        test_loader = None

    # Wypisz rozkład klas (istotne przy skośnym datasecie)
    if isinstance(train_dataset_resnet, Subset):
        targets = [train_dataset_resnet.dataset.targets[i] for i in train_dataset_resnet.indices]
    else:
        targets = train_dataset_resnet.targets
    counts = np.bincount(targets)
    print(f"Rozkład klas — min: {counts.min()}, max: {counts.max()}, "
          f"ratio: {counts.max()/counts.min():.1f}x")
    

    return train_dataset_resnet, train_dataset_dino, train_dataset_scratch, val_loader, test_loader


train_dataset_resnet, train_dataset_dino, train_dataset_scratch, val_loader, test_loader = load_datasets(_cfg, DATA_PATH, SEED, BATCH_SIZE)


Train :  18330 próbek
Val   :   2000 próbek
Test  :   2000 próbek
Rozkład klas — min: 537, max: 4600, ratio: 8.6x


### Funkcje pomocnicze

In [6]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


def get_scratch_model(num_classes: int, device: torch.device) -> nn.Module:
    """
    ResNet18 z LOSOWO ZAINICJOWANYMI wagami.
    Wszystkie parametry są trenowalne — brak zamrożonych warstw.
    """
    model = models.resnet18(weights=None)  # weights=None → brak pretrainingu
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(device)


# --- Ekstrakcja embeddingów DINO (do klasteryzacji) ---

def get_dino_embeddings(dataset, batch_size=128):
    """Oblicza embeddingi DINOv2-large na żywo z datasetu."""
    extractor = AutoModel.from_pretrained("facebook/dinov2-large")
    extractor.eval().to(device)
    all_embeddings = []
    with torch.inference_mode(), torch.amp.autocast("cuda", enabled=device.type == "cuda"):
        for imgs, _ in tqdm(
            DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0),
            desc="DINO feature extraction",
        ):
            emb = extractor(imgs.to(device)).last_hidden_state[:, 0, :]
            all_embeddings.append(emb.cpu())
    return torch.cat(all_embeddings).to(device)


def load_dino_embeddings_from_pkl(pkl_path: str, device: torch.device) -> torch.Tensor:
    """Wczytuje wcześniej obliczone embeddingi DINO z pliku .pkl."""
    import pickle
    print(f"Wczytywanie embeddingów z: {pkl_path}")
    data = torch.load(pkl_path, map_location=device)
    # Obsługa zarówno samego tensora, jak i słownika {"embeddings": tensor}
    if isinstance(data, dict):
        tensor = data.get("embeddings", data.get("dino_embeddings", next(iter(data.values()))))
    else:
        tensor = data
    if not isinstance(tensor, torch.Tensor):
        tensor = torch.tensor(tensor)
    print(f"Wczytano embeddingi: shape={tuple(tensor.shape)}, dtype={tensor.dtype}")
    return tensor.float().to(device)


def plot_training_curve(train_losses, val_losses, title="Training Curve"):
    plt.figure(figsize=(8, 5))
    epochs = range(1, len(train_losses) + 1)
    plt.plot(epochs, train_losses, marker="o", markersize=3, label="Train Loss")
    plt.plot(epochs, val_losses,   marker="s", markersize=3, label="Val Loss")
    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.show()


### Pętla treningowa

In [7]:
def train_model(
    model, criterion, optimizer, scheduler,
    train_loader, val_loader, test_loader,
    device, title,
    max_epochs=MAX_EPOCHS,
    patience=EARLY_STOPPING_PATIENCE,
    epoch_tqdm=True,
):
    print(f"\n[TRENING] Start: {title}")
    use_amp = device.type == "cuda"
    scaler  = torch.amp.GradScaler("cuda", enabled=use_amp)

    train_losses, val_losses = [], []
    best_val_loss    = float("inf")
    best_state       = {k: v.clone() for k, v in model.state_dict().items()}
    patience_counter = 0

    epoch_iter = tqdm(range(max_epochs), desc="Epochs") if epoch_tqdm else range(max_epochs)

    for epoch in epoch_iter:
        # --- trening ---
        model.train()
        running_loss = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=use_amp):
                loss = criterion(model(imgs), labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item()
            scheduler.step()
        train_losses.append(running_loss / len(train_loader))

        # --- walidacja ---
        model.eval()
        val_loss = 0.0
        with torch.inference_mode():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                val_loss += criterion(model(imgs), labels).item()
        avg_val_loss = val_loss / len(val_loader)
        val_losses.append(avg_val_loss)

        # if scheduler is not None:
        #     scheduler.step()

        # --- early stopping ---
        if avg_val_loss < best_val_loss:
            best_val_loss    = avg_val_loss
            patience_counter = 0
            best_state       = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping przy epoce {epoch + 1}")
                break

    model.load_state_dict(best_state)
    model.eval()

    # --- ewaluacja końcowa ---
    if val_loader is not None and test_loader is not None:
        results = {}
        with torch.inference_mode():
            for loader, split_name in [(val_loader, "Validation"), (test_loader, "Test")]:
                correct, total = 0, 0
                for imgs, labels in loader:
                    imgs, labels = imgs.to(device), labels.to(device)
                    _, predicted = torch.max(model(imgs), 1)
                    total   += labels.size(0)
                    correct += (predicted == labels).sum().item()
                acc = 100 * correct / total
                results[split_name] = acc
                print(f"[RESULT] {title} — {split_name} Accuracy: {acc:.2f}%")

    return results["Test"], train_losses, val_losses


### Ekstrakcja cech i hierarchiczny k-means

In [11]:
# ── KOMÓRKA 14: Ładowanie embeddingów i klastrowanie ────────────────────────
import torch.nn.functional as F
import json

print("\n--- Embeddingi DINO ---")
set_seed(SEED)

if LOAD_DINO_EMBEDDINGS_FROM_PKL:
    dino_embeddings = torch.load(DINO_EMBEDDINGS_PKL_PATH, map_location=device).float()
else:
    dino_embeddings = get_dino_embeddings(train_dataset_dino).float()
    torch.save(dino_embeddings.cpu(), DINO_EMBEDDINGS_PKL_PATH)
    print(f"Zapisano embeddingi do: {DINO_EMBEDDINGS_PKL_PATH}")

dino_embeddings_np = dino_embeddings.cpu().numpy()
dino_embeddings_norm = F.normalize(dino_embeddings, p=2, dim=1)
dino_embeddings_np_norm = dino_embeddings_norm.cpu().numpy()


# specjalnie pliki na odwrót, są źle nazwane
with open(os.path.join(PROJECT_PATH, "best_agl_clustering.json")) as f:
    cfg_kmeans = json.load(f)
with open(os.path.join(PROJECT_PATH, "best_kmeans_clustering.json")) as f:
    cfg_agl = json.load(f)

print("\n--- Uruchamianie klastrowania ---")
set_seed(42)
kmeans_labels = KMeans(
    n_clusters=cfg_kmeans["params"]["n_clusters"],
    n_init="auto",
    random_state=42,
).fit_predict(dino_embeddings_np_norm)
print(f"KMeans: {len(set(kmeans_labels))} klastrów")

set_seed(42)
agl_labels = AgglomerativeClustering(
    n_clusters=cfg_agl["params"]["n_clusters"],
    linkage=cfg_agl["params"].get("linkage"),
).fit_predict(dino_embeddings_np_norm)
print(f"Agglomerative: {len(set(agl_labels))} klastrów")

set_seed(42)
print("--- Hierarchiczny k-means (DINO) ---")
clusters_dino = hkmg.hierarchical_kmeans_with_resampling(
    data=dino_embeddings,
    n_clusters=N_CLUSTERS_DINO,
    n_levels=N_LEVELS_DINO,
    sample_sizes=SAMPLE_SIZES_DINO,
    verbose=False,
)
cl_dino = HierarchicalCluster.from_dict(clusters_dino)
print("Klasteryzacja zakończona.")

def repetetive_hierarchical_sampling(ts):
    set_seed(42)
    return hs.hierarchical_sampling(cl_dino, target_size=ts)

sampling_methods = [
    ("kmeans",        lambda ts: _sample_from_labels(kmeans_labels, ts, 42)),
    ("agglomerative", lambda ts: _sample_from_labels(agl_labels, ts, 42)),
    ("random",        lambda ts: np.random.default_rng(42).choice(dino_embeddings_np_norm.shape[0], size=ts, replace=False)),
    ("hierarchical",  lambda ts: repetetive_hierarchical_sampling(ts)),
]

sampling_methods = [
    item for item in sampling_methods if item[0] in SAMPLING_METHODS
]




--- Embeddingi DINO ---


C:\Users\Kacper\AppData\Local\Temp\ipykernel_3160\393034613.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  dino_embeddings = torch.load(DINO_EMBEDDINGS_PKL_PATH, map_lo


--- Uruchamianie klastrowania ---


KeyboardInterrupt: 

### Główna pętla eksperymentu


In [9]:
print("Eksperyment: Clustering-based Sampling | Model od Zera (multi-seed)")
print(f"Dataset: {DATASET}  (α={_cfg['alpha']})")
print(f"Seedy: {SEEDS}")
print("="*60)

total_samples = len(train_dataset_scratch)

MODEL_DIR = Path("models_percentage")
MODEL_DIR.mkdir(exist_ok=True)

print("Eksperyment: Sampling Methods | Model od Zera (multi-seed)")
print(f"Dataset: {DATASET}  (α={_cfg['alpha']})")
print(f"Seedy: {SEEDS}")
print(f"Metody: {[m[0] for m in sampling_methods]}")
print("=" * 60)

results_df = pd.DataFrame(
    columns=["sampling_method", "embeddings", "percentage", "seed", "test_accuracy", "train_losses", "val_losses"]
)


for method_name, get_idx in sampling_methods:
    for p in percentages:
        pct_label = f"{int(p * 100)}%"
        target_size = int(total_samples * p)

        for seed in SEEDS:
            print(f"\n{'=' * 60}")
            print(f"Metoda: {method_name} | Subset: {pct_label} ({target_size} próbek) | Seed: {seed}")
            if  Path(MODEL_DIR / f"{method_name}_{pct_label}_seed{seed}.pt").exists():
                continue

            
            idx = get_idx(target_size)
            current_ds = Subset(train_dataset_scratch, idx)

            set_seed(seed)
            model     = get_scratch_model(NUM_CLASSES, device)
            criterion = nn.CrossEntropyLoss()

            train_loader_current = DataLoader(
                current_ds, batch_size=BATCH_SIZE, shuffle=True,
                num_workers=4, pin_memory=True, persistent_workers=True,
                generator=torch.Generator().manual_seed(seed),
            )

            optimizer = optim.SGD(
                model.parameters(),
                lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY,
            )
            scheduler = optim.lr_scheduler.OneCycleLR(
                optimizer, max_lr=LR, epochs=MAX_EPOCHS, steps_per_epoch=len(train_loader_current),
            )
            # !!!!!!!!!!!!!!!!! ZMIEN MAX EPOCHS !!!!!!!!!!!!!!! TO DO
            test_acc, train_losses, val_losses = train_model(
                model, criterion, optimizer, scheduler,
                train_loader_current, val_loader, test_loader,
                device,
                title=f"{method_name} | {pct_label} | seed={seed}",
                max_epochs=MAX_EPOCHS,
                patience=EARLY_STOPPING_PATIENCE,
                epoch_tqdm=True,
            )

            plot_training_curve(
                train_losses, val_losses,
                title=(
                    f"Scratch ResNet18 | Dino embeddings\n"
                    f"{method_name} ({pct_label}) seed={seed} | Test acc: {test_acc:.2f}%"
                ),
            )

            model_path = MODEL_DIR / f"{method_name}_{pct_label}_seed{seed}.pt"
            torch.save(model.state_dict(), model_path)

            results_df.loc[len(results_df)] = {
                "sampling_method": method_name,
                "embeddings":      "Dino",
                "percentage":      pct_label,
                "seed":            seed,
                "test_accuracy":   test_acc,
                "train_losses":    train_losses,
                "val_losses":      val_losses,
            }

            results_df.to_pickle(f"scratch_results_{method_name}_{p}_multiseed.pkl")
            print(f"  Seed {seed} -> Test acc: {test_acc:.2f}%")

print("\nWszystkie eksperymenty zakończone.")

summary = (
    results_df
    .groupby(["sampling_method", "percentage"])["test_accuracy"]
    .agg(["mean", "std", "count"])
    .reset_index()
)
summary["result"] = summary.apply(lambda r: f"{r['mean']:.2f} ± {r['std']:.2f}", axis=1)
print("\n", summary[["sampling_method", "percentage", "result", "count"]].to_string(index=False))

Eksperyment: Clustering-based Sampling | Model od Zera (multi-seed)
Dataset: alpha_10_seed_129  (α=1)
Seedy: [45390, 12345, 98765, 1081]
Eksperyment: Sampling Methods | Model od Zera (multi-seed)
Dataset: alpha_10_seed_129  (α=1)
Seedy: [45390, 12345, 98765, 1081]
Metody: ['kmeans', 'agglomerative', 'random', 'hierarchical_2']

Metoda: kmeans | Subset: 30% (5499 próbek) | Seed: 45390

Metoda: kmeans | Subset: 30% (5499 próbek) | Seed: 12345

Metoda: kmeans | Subset: 30% (5499 próbek) | Seed: 98765

Metoda: kmeans | Subset: 30% (5499 próbek) | Seed: 1081

Metoda: kmeans | Subset: 50% (9165 próbek) | Seed: 45390

Metoda: kmeans | Subset: 50% (9165 próbek) | Seed: 12345

Metoda: kmeans | Subset: 50% (9165 próbek) | Seed: 98765

Metoda: kmeans | Subset: 50% (9165 próbek) | Seed: 1081

Metoda: kmeans | Subset: 70% (12831 próbek) | Seed: 45390

Metoda: kmeans | Subset: 70% (12831 próbek) | Seed: 12345

Metoda: kmeans | Subset: 70% (12831 próbek) | Seed: 98765

Metoda: kmeans | Subset: 70% (1

In [10]:
def test_model(model, test_loader=test_loader):
    model.eval()
    all_count = 0
    predicted_count = 0
    with torch.inference_mode():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            _, predicted = torch.max(model(images),1)
            predicted_count += sum(predicted==labels)
            all_count += len(labels)

    return (predicted_count/all_count).cpu()


arrs = [[],[],[],[]]
for path in tqdm(Path(r"C:\Users\Kacper\Documents\iad 6 sem\wb2\WB2_Project\models_percentage").rglob("*.pt"), total =64):
    if path.name.split("_")[1]=="2":
        continue
    name, percentage = path.name.split("_")[0], path.name.split("_")[1]
    seed = int(path.name.split("_")[2][4:-3])
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, 10)
    model.load_state_dict(torch.load(path))
    model.to(device)
    arrs[0].append(name)
    arrs[1].append(percentage)
    arrs[2].append(test_model(model))
    arrs[3].append(seed)

df = pd.DataFrame({
    "sampling_method": arrs[0],
    "percentage": arrs[1],
    "seed": arrs[3],
    "test_accuracy": arrs[2],
})

means = df.groupby(["sampling_method","percentage"])["test_accuracy"].mean()
stds = df.groupby(["sampling_method","percentage"])["test_accuracy"].std()

table = means.combine(stds, lambda m, s: f"{m:.3f}({s:.3f})").unstack("percentage")
table

  0%|          | 0/64 [00:00<?, ?it/s]

C:\Users\Kacper\AppData\Local\Temp\ipykernel_3160\2972715914.py:23: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(path))


percentage,30%,50%,70%,90%
sampling_method,,,,
agglomerative,0.660(0.018),0.675(0.029),0.691(0.018),0.725(0.019)
hierarchical,0.625(0.020),0.687(0.007),0.696(0.033),0.726(0.007)
kmeans,0.647(0.017),0.668(0.012),0.693(0.011),0.707(0.035)
random,0.620(0.027),0.666(0.026),0.686(0.047),0.720(0.011)
